# AutoCompose

## Import Section

In [1]:
import os
os.chdir("C:/Users/KTS-WS-2501/Documents/Composer-py")
os.getcwd()

'C:\\Users\\KTS-WS-2501\\Documents\\Composer-py'

In [2]:
import torch
torch.manual_seed(64)

In [3]:
import json
import os

In [4]:
from torch.optim import AdamW
from torch.utils.data import Dataset, random_split, DataLoader, RandomSampler, SequentialSampler
from transformers import GPT2LMHeadModel, GPT2Tokenizer

c:\Users\KTS-WS-2501\Documents\composer-py\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load tokenizer and model

In [6]:
model_name = "gpt2"

In [7]:
bos_token = "<|startoftext|>"
eos_token = "<|endoftext|>"
pad_token = "<|pad|>"

In [8]:
# tokenizer = GPT2Tokenizer.from_pretrained(
#     model_name, 
#     bos_token=bos_token, 
#     eos_token=eos_token, 
#     pad_token=pad_token
# )

In [9]:
tokenizer = GPT2Tokenizer.from_pretrained(model_name,bos_token=bos_token,eos_token=eos_token,pad_token=pad_token)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 13579.14it/s]


## Data Preparation

In [10]:
data_dir = "data"
file_path = "anticipation.json"

In [11]:
file_full_path = os.path.normpath(os.path.join(data_dir,file_path))

In [12]:
with open(file_full_path, "r") as f:
    data = json.load(f)

data[:5]

[{'poem': 'Though the birds sang gayly to him,\nThough the wild-flowers of the meadow\nFilled the air with odors for him;\nThough the forests and the rivers\nSang and shouted at his coming,\nStill his heart was sad within him,\nFor he was alone in heaven.',
  'id': 26},
 {'poem': 'Rise up from your bed of branches,\nRise, O youth, and wrestle with me!"\nFaint with famine, Hiawatha\nStarted from his bed of branches,\nFrom the twilight of his wigwam\nForth into the flush of sunset\nCame, and wrestled with Mondamin;\nAt his touch he felt new courage\nThrobbing in his brain and bosom,\nFelt new life and hope and vigor\nRun through every nerve and fibre.',
  'id': 93},
 {'poem': 'On the morrow and the next day,\nWhen the sun through heaven descending,\nLike a red and burning cinder\nFrom the hearth of the Great Spirit,\nFell into the western waters,\nCame Mondamin for the trial,\nFor the strife with Hiawatha;\nCame as silent as the dew comes,\nFrom the empty air appearing,\nInto empty air r

In [13]:
data[0].keys()

dict_keys(['poem', 'id'])

In [14]:
tokenizer("hello",truncation=True,max_length=10,padding="max_length")

{'input_ids': [31373, 50258, 50258, 50258, 50258, 50258, 50258, 50258, 50258, 50258], 'attention_mask': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [18]:
tokenizer.eos_token

'<|endoftext|>'

In [27]:
class PoemDataLoader(Dataset):
    def __init__(self, poems, tokenizer, max_len=768):
        self.attn_mask = []
        self.input_ids = []

        for poem in poems:
            
            encodings_dict = tokenizer("<|startoftext|>"+poem+"<|endoftext|>",max_length=max_len,truncation=True,padding="max_length") 
            self.input_ids.append(torch.tensor(encodings_dict["input_ids"]))
            self.attn_mask.append(torch.tensor(encodings_dict["attention_mask"]))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self,idx):
        return self.input_ids[idx], self.attn_mask[idx]

In [24]:
poems_text = [poem["poem"] for poem in data]
len(poems_text)

28956

In [28]:
poem_data_loader = PoemDataLoader(poems=poems_text,max_len=768,tokenizer=tokenizer)

In [37]:
y = [len(tokenizer.encode(poem["poem"])) - len(poem["poem"].split()) for poem in data]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1198 > 1024). Running this sequence through the model will result in indexing errors
